> 在线讲义：[第 15 章 · sin 和 cos：圆上转圈](https://wikieden.github.io/transformer-visual/lessons/15-sin-cos.html) · 本书《从零读懂 Transformer》

# 第 15 章 · sin 和 cos：圆上转圈

配套讲义：[在线版](https://wikieden.github.io/transformer-visual/lessons/15-sin-cos.html) · 仓库里 `transformer-deep-dive/notes/lessons/15-sin-cos.md`

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 torch 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。不需要 GPU。

In [ ]:
import torch

torch.set_printoptions(precision=3, sci_mode=False)  # 打印保留 3 位小数，不用科学计数法
print("torch", torch.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一行代码、跑出来是什么。

| 讲义 | 公式 / 说法 | PyTorch 写法 | 跑出来 |
|---|---|---|---|
| 圈 → 弧度 | 1 圈 = 2π | `turns * 2 * math.pi` | ¼ 圈 = 1.57 |
| sin 是高度 | 0、¼、½、¾、1 圈 | `torch.sin(rad)` | [0, 1, 0, −1, 0, …] |
| cos 是左右 | 同上 | `torch.cos(rad)` | [1, 0, −1, 0, 1, …] |
| 转一圈就重复 | sin(1¼ 圈) = sin(¼ 圈) | `math.isclose` | True |
| (cos, sin) 号码牌 | 每座多转 ¼ 圈 | `seat_tag(seat)` | 3 号 (0, −1)；1000 号 (1, 0) |

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import math
import torch

**函数 `turns_to_radians`**：转了几圈 → 沿边走了多远：1 圈 = 2π 弧度。torch.sin / cos 只认弧度。

In [ ]:
def turns_to_radians(turns: torch.Tensor) -> torch.Tensor:
    """转了几圈 → 沿边走了多远：1 圈 = 2π 弧度。torch.sin / cos 只认弧度。"""
    return turns * 2 * math.pi

**函数 `seat_tag`**：第四步的号码牌：每个座位多转 1/4 圈，牌 = (cos, sin)。

In [ ]:
def seat_tag(seat: int, turns_per_seat: float = 0.25) -> torch.Tensor:
    """第四步的号码牌：每个座位多转 1/4 圈，牌 = (cos, sin)。"""
    angle = turns_to_radians(torch.tensor(seat * turns_per_seat, dtype=torch.float64))
    return torch.stack([torch.cos(angle), torch.sin(angle)])

**函数 `r`**：小工具，把结果整理成好读的样子再打印。

In [ ]:
def r(t: torch.Tensor, n: int = 2) -> list[float]:
    return [round(float(x), n) + 0.0 for x in t]  # + 0.0 把 -0.0 变成 0.0

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
turns = torch.tensor([0, 0.25, 0.5, 0.75, 1, 1.25, 1.5, 1.75, 2], dtype=torch.float64)
rad = turns_to_radians(turns)
print("圈  :", turns.tolist())
print("弧度:", r(rad))
print("sin :", r(torch.sin(rad)))
print("cos :", r(torch.cos(rad)))
print("转一圈就重复：sin(1¼ 圈) == sin(¼ 圈):", math.isclose(math.sin(rad[5]), math.sin(rad[1])))
x = torch.linspace(-1000, 1000, 100001)
print("sin、cos 永远在 −1 到 1:", float(torch.sin(x).min()), float(torch.sin(x).max()))

In [ ]:
print("\n(cos, sin) 号码牌，狗 = [2, 0]")
dog = torch.tensor([2.0, 0.0], dtype=torch.float64)
for seat in [1, 2, 3, 4, 1000]:
    tag = seat_tag(seat)
    print(f"  {seat:>4} 号座: 牌 {r(tag)}  狗 + 牌 = {r(dog + tag)}")

In [ ]:
print("\nsin(1) =", round(math.sin(1), 2), "（沿边走 1，约", round(1 / (2 * math.pi), 2), "圈）")

## 3. 代码里才看得到的

1. **`torch.sin`、`torch.cos` 只认弧度**：写 `torch.sin(90)` 得到的不是 1，而是 sin(90 弧度) ≈ 0.89。要从"圈"或"度"换成弧度再算，这正是第五步讲的。
2. **打印出 −0.0**：cos(¾ 圈) 算出来是 −0.00000000000000018 这种极小的数（float 存不下精确的 π），四舍五入后带着负号。代码里 `+ 0.0` 把它变回 0.0，不影响任何结果。

## 4. 自己改一改

- 把每座多转的 ¼ 圈改成 1/3 圈，第几号座会和 1 号座撞牌？

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> sin 管高低，cos 管左右；
> 转圈不出正负一，一圈一重复；
> 一圈等于 2π，括号里是走了多远。